# 504日协议修订与人工容量复核
本笔记本仅验证用户选择、协议字节和人工成交计数；不估计市场收益。两个代码单元由项目Python进程依序执行并捕获实际输出，没有使用Jupyter内核。

In [1]:
import hashlib, json
from pathlib import Path
import pandas as pd
root=Path('/Users/guo/Documents/stocks/quant-research')
art=root/'artifacts/earnings-validation-20261012'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
revision=json.loads((art/'protocol-revision-v2.json').read_text())
for kind in ['base_protocol','current_protocol']:
    assert sha(root/revision[kind]['path'])==revision[kind]['sha256']
design=json.loads((root/revision['current_protocol']['path']).read_text())
assert design['validation']['single_formal_look_sessions']==504
assert design['validation']['minimum_mature_unique_company_report_events']==100
assert design['validation']['minimum_active_entry_dates']==60
assert design['costs']['broker_commission_rate']==.0005
print(json.dumps({'formal_look':504,'minimum_events':100,'minimum_entry_dates':60,'old_protocol_bytes_preserved':True}))

{"formal_look": 504, "minimum_events": 100, "minimum_entry_dates": 60, "old_protocol_bytes_preserved": true}


In [2]:
audit_root=art/'statistics-audit-v3'
audit=json.loads((audit_root/'audit.json').read_text())
for relative, expected in audit['source_hashes'].items():
    assert sha(root/relative)==expected
for relative, expected in audit['output_hashes'].items():
    assert sha(audit_root/relative)==expected
calendar=pd.read_csv(audit_root/'expected-artificial-calendar.csv').date.tolist()
fills=pd.read_parquet(audit_root/'base/treated/fills.parquet')
buys=fills.loc[fills.side.eq('buy')]
mature=buys.loc[buys.date.map(lambda d: calendar.index(d)+20<len(calendar))]
assert len(calendar)==504 and len(buys)==156 and len(mature)==150
assert mature.date.nunique()==25
assert not audit['statistics']['numerical_gate_results']['mature_active_entry_dates']
assert not audit['formal_profit_qualified']
print(json.dumps({'artificial_observed_sessions':len(calendar),'buy_fills':len(buys),'mature_company_report_fills':len(mature),'mature_entry_dates':int(mature.date.nunique()),'formal_profit_qualified':False}))

{"artificial_observed_sessions": 504, "buy_fills": 156, "mature_company_report_fills": 150, "mature_entry_dates": 25, "formal_profit_qualified": false}
